# Procedural Frozen Lake with the Q* expert policy

Procedural Frozen Lake is configured with a **value-iteration** expert. The wrapper stack injects optimal Q-values into `metadata["q_star"]` each step so you can derive actions without a separate policy network.

## Imports

Discrete actions are passed as `TensorDict` with an `action.discrete` field (one integer per env).

In [ ]:
import numpy as np
import torch
from tensordict import TensorDict

from mouse.envs import EnvConfig, make_vector_env

## Environment

In [ ]:
cfg = EnvConfig.procedural_frozenlake(seed=7, num_envs=1)
env = make_vector_env(cfg)

## Expert rollout

1. The first `step` resets and returns `metadata["q_star"]` for the start state.
2. Take `argmax` over Q* to pick greedy actions.
3. Pass those actions on the next `step`.

In [ ]:
episodes = 0
data, metadata, metrics = env.step(env.sample_random_actions())  # initial reset

for step in range(200):
    q_star = metadata["q_star"]  # (num_envs, num_actions)
    raw_actions = np.argmax(q_star, axis=-1)

    actions = [
        TensorDict(
            {"action": {"discrete": torch.tensor([int(raw_actions[i])], dtype=torch.int64)}},
            batch_size=[],
        )
        for i in range(env.num_envs)
    ]
    data, metadata, metrics = env.step(actions)

    td = data[0]
    if td["done"].item() != 0:
        episodes += 1
        done_code = td["done"].item()
        outcome = "terminated" if done_code == 1 else "truncated"
        print(
            f"step={step:3d}  episode={episodes}  outcome={outcome}  "
            f"reward={td['reward'].item():.1f}"
        )

## Cleanup

In [ ]:
env.close()